# CSMS AI Accuracy Improvement v2 — Shared Criteria + Confidence Calibration

**Goal:** Improve juror confidence and accuracy through shared criterion fact sheets, context-expanded KB interpretation, and calibrated confidence scoring.

## Key Improvements over v1 (notebook 03):
1. **Shared criterion fact sheet** — single authoritative fact sheet per category shared across all jurors
2. **Context-expanded KB interpretation** — AI expands understanding of validation_rules and scoring_rules beyond explicit text
3. **Confidence calibration** — calibrated confidence with 0.8 threshold for review flagging
4. **Evidence grounding** — every criterion linked to specific chunks and source files
5. **Juror agreement analysis** — measure consensus quality and identify disagreements

## Scoring Logic:
- **Juror scores:** Continuous (0.0 to 10.0)
- **Final score:** Upward rounding
  - `0` = no evidence
  - `(0, 3]` → `3`
  - `(3, 6]` → `6`
  - `(6, 10]` → `10`
- **Confidence threshold:** `0.8`
  - If `confidence < 0.8`: flag for review
  - No automatic downgrade

## Outputs:
- Criterion-level audit with confidence scores
- Shared fact sheets per category
- Context-expanded KB annotations
- Juror agreement analysis
- HTML review dashboard with confidence indicators

---

## 1. Setup & Configuration

In [ ]:
import os
import sys
import json
import re
import sqlite3
import struct
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from IPython.display import display, Markdown, HTML

sys.path.append(os.path.abspath('..'))

PROJECT_ROOT = os.path.abspath('..')
MARINE_DIR = os.path.join(PROJECT_ROOT, 'test_submissions', 'marine')
KB_DIR = os.path.join(PROJECT_ROOT, 'knowledge_base', 'marine')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'notebook_v2', 'outputs')
DB_DIR = os.path.join(OUTPUT_DIR, 'databases')
CALIBRATION_DIR = os.path.join(KB_DIR, 'calibration')
REVIEW_DIR = os.path.join(OUTPUT_DIR, 'review_v2')

os.makedirs(CALIBRATION_DIR, exist_ok=True)
os.makedirs(REVIEW_DIR, exist_ok=True)

BASELINE_RESULTS = os.path.join(OUTPUT_DIR, 'reports', 'batch_evaluation_results.json')
V1_RESULTS = os.path.join(OUTPUT_DIR, 'review', 'criterion_level_audit.json')

print(f'Project root: {PROJECT_ROOT}')
print(f'Marine dir:   {MARINE_DIR}')
print(f'KB dir:       {KB_DIR}')
print(f'Calibration:  {CALIBRATION_DIR}')
print(f'Review v2:    {REVIEW_DIR}')
print('\nSetup complete.')

In [ ]:
TARGET_CATEGORIES = None
USE_LARGE_DB = True
ENABLE_LLM_EXTRACTION = True
CONFIDENCE_THRESHOLD = 0.8
VERBOSE = True

if TARGET_CATEGORIES is None:
    print('Processing all 29 categories')
else:
    print(f'Processing {len(TARGET_CATEGORIES)} categories: {TARGET_CATEGORIES}')

print(f'Using database: {"large" if USE_LARGE_DB else "small"}')
print(f'LLM extraction: {ENABLE_LLM_EXTRACTION}')
print(f'Confidence threshold: {CONFIDENCE_THRESHOLD}')

## 2. Load Baseline Results

In [ ]:
with open(BASELINE_RESULTS, 'r', encoding='utf-8') as f:
    baseline = json.load(f)

with open(V1_RESULTS, 'r', encoding='utf-8') as f:
    v1_results = json.load(f)

baseline_by_id = {r['category_id']: r for r in baseline['results']}
v1_by_id = {r['category_id']: r for r in v1_results['audits']}

print(f'Loaded baseline: {baseline["total_categories"]} categories')
print(f'Baseline original accuracy: {baseline["summary"]["original"]["accuracy"]:.1f}%')
if 'enhanced_large' in baseline['summary']:
    print(f'Baseline enhanced-large accuracy: {baseline["summary"]["enhanced_large"]["accuracy"]:.1f}%')

print(f'\nLoaded v1 results: {v1_results["total_categories"]} categories')
print(f'V1 calibrated accuracy: {v1_results["summary"]["calibrated_accuracy"]:.1f}%')
print(f'V1 improvement: +{v1_results["summary"]["improvement"]:.1f}%')

## 3. Discover Categories & Load Knowledge Base

In [ ]:
def discover_categories(marine_dir: str) -> List[Dict[str, Any]]:
    categories = []
    pattern = re.compile(r'^([0-9]+(\.[0-9]+)?)')
    
    for item in sorted(os.listdir(marine_dir)):
        full_path = os.path.join(marine_dir, item)
        if os.path.isdir(full_path):
            m = pattern.match(item)
            if m:
                cat_id = m.group(1)
                expected_file = os.path.join(full_path, 'expected_result.json')
                expected_score = None
                if os.path.exists(expected_file):
                    with open(expected_file, 'r', encoding='utf-8') as f:
                        expected_score = json.load(f).get('expected_score')
                
                categories.append({
                    'id': cat_id,
                    'name': item,
                    'path': full_path,
                    'expected_score': expected_score
                })
    
    def sort_key(x):
        parts = x['id'].split('.')
        return [int(p) for p in parts]
    
    categories.sort(key=sort_key)
    return categories

def load_kb(category_id: str) -> Dict[str, Any]:
    clean_id = re.match(r'^[0-9]+(\.[0-9]+)?', str(category_id).strip())
    clean_id = clean_id.group(0) if clean_id else str(category_id).strip()
    result = {}
    for kind, sub, prefix in [('category', 'categories', ''), ('scoring_rules', 'scoring_rules', 'SR-'), ('validation_rules', 'validation_rules', 'VR-'), ('document_requirements', 'document_requirements', 'DR-')]:
        fpath = os.path.join(KB_DIR, sub, f'{prefix}{clean_id}.json')
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                result[kind] = json.load(f)
        else:
            result[kind] = {}
    return result

categories = discover_categories(MARINE_DIR)

if TARGET_CATEGORIES:
    categories = [c for c in categories if c['id'] in TARGET_CATEGORIES]

print(f'Discovered {len(categories)} categories')
for cat in categories[:5]:
    print(f'  {cat["id"]}: {cat["name"]} (expected: {cat["expected_score"]})')
if len(categories) > 5:
    print(f'  ... and {len(categories) - 5} more')

## 4. Database Functions

In [ ]:
def blob_to_embedding(blob: bytes) -> List[float]:
    n = len(blob) // 4
    return list(struct.unpack(f'{n}f', blob))

def load_category_chunks(category_id: str, use_large: bool = True) -> List[Dict[str, Any]]:
    db_suffix = 'large' if use_large else 'small'
    db_name = f'cat_{category_id.replace(".", "_")}_{db_suffix}.db'
    db_path = os.path.join(DB_DIR, db_name)
    
    if not os.path.exists(db_path):
        return []
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    cursor.execute('''
        SELECT chunk_id, category_id, filename, doc_type, tier, priority, 
               evidence_idx, chunk_idx, text, char_count, embedding
        FROM chunks
        ORDER BY priority, filename, evidence_idx, chunk_idx
    ''')
    
    rows = cursor.fetchall()
    conn.close()
    
    chunks = []
    for row in rows:
        chunks.append({
            'chunk_id': row[0],
            'category_id': row[1],
            'filename': row[2],
            'doc_type': row[3],
            'tier': row[4],
            'priority': row[5],
            'evidence_idx': row[6],
            'chunk_idx': row[7],
            'text': row[8],
            'char_count': row[9],
            'has_embedding': row[10] is not None
        })
    
    return chunks

print('Database functions loaded.')

## 5. Context-Expanded KB Interpretation

Expand understanding of validation_rules and scoring_rules beyond explicit text. Infer implicit requirements, regulatory context, and marine-specific terminology.

In [ ]:
def expand_kb_context(category_id: str, kb: Dict[str, Any]) -> Dict[str, Any]:
    """
    Expand KB interpretation with implicit requirements and marine context.
    """
    context = {
        'category_id': category_id,
        'explicit_requirements': {
            'keywords': kb.get('validation_rules', {}).get('ai_checkpoint_keywords', []),
            'mandatory_fields': kb.get('validation_rules', {}).get('mandatory_fields', []),
            'validation_methods': kb.get('validation_rules', {}).get('validation_methods', [])
        },
        'implicit_requirements': {
            'regulatory_context': [],
            'marine_terminology': [],
            'document_patterns': [],
            'implementation_signals': []
        },
        'score_gates': {}
    }
    
    sr = kb.get('scoring_rules', {})
    score_levels = sr.get('score_levels', {})
    
    # Infer regulatory context from category ID and title
    cat_info = kb.get('category', {})
    title = cat_info.get('title', '').lower()
    desc = cat_info.get('description', '').lower()
    
    # Marine regulatory mappings
    marine_mappings = {
        'sertifikasi': ['ISO', 'certification', 'akreditasi'],
        'dokumen': ['document', 'prosedur', 'SOP', 'manual'],
        'pelatihan': ['training', 'kompetensi', 'sertifikat pelatihan'],
        'risiko': ['HIRADC', 'risk assessment', 'hazard identification'],
        'insiden': ['incident', 'kecelakaan', 'investigasi'],
        'audit': ['audit', 'inspeksi', 'pemeriksaan'],
        'tanggap darurat': ['emergency', 'ERP', 'drill', 'simulasi'],
        'kontraktor': ['subkontraktor', 'vendor', 'mitra kerja'],
        'lingkungan': ['environmental', 'MARPOL', 'limbah', 'waste'],
        'keselamatan': ['safety', 'SOLAS', 'ISPS', 'STCW']
    }
    
    for key, terms in marine_mappings.items():
        if key in title or key in desc:
            context['implicit_requirements']['regulatory_context'].extend(terms)
            context['implicit_requirements']['marine_terminology'].extend(terms)
    
    # Infer document patterns from scoring rules
    for score_level in ['0', '3', '6', '10']:
        level_desc = score_levels.get(score_level, '')
        if 'DOKUMEN' in level_desc or 'dokumen' in level_desc.lower():
            # Extract document requirements
            lines = level_desc.split('\n')
            for line in lines:
                if line.strip().startswith('-') or line.strip().startswith('•'):
                    doc_req = line.strip().lstrip('-•').strip()
                    if doc_req and len(doc_req) > 10:
                        context['implicit_requirements']['document_patterns'].append(doc_req)
    
    # Infer implementation signals from validation methods
    validation_methods = kb.get('validation_rules', {}).get('validation_methods', [])
    if 'date_validation' in validation_methods:
        context['implicit_requirements']['implementation_signals'].append('temporal_evidence')
    if 'signature_check' in validation_methods:
        context['implicit_requirements']['implementation_signals'].append('authorization_evidence')
    if 'keyword_search' in validation_methods:
        context['implicit_requirements']['implementation_signals'].append('compliance_keywords')
    if 'table_check' in validation_methods:
        context['implicit_requirements']['implementation_signals'].append('structured_data')
    
    # Build score gates from scoring rules
    for score_level in ['0', '3', '6', '10']:
        level_desc = score_levels.get(score_level, '')
        gate = {
            'description': level_desc[:200],
            'conditions': [],
            'required_documents': [],
            'required_evidence': []
        }
        
        # Extract conditions from AI DECISION lines
        for line in level_desc.split('\n'):
            if 'AI DECISION' in line or 'AI:' in line:
                gate['conditions'].append(line.strip())
            elif 'SYARAT WAJIB' in line or 'semua harus terpenuhi' in line:
                gate['conditions'].append('ALL_MANDATORY_REQUIRED')
        
        context['score_gates'][score_level] = gate
    
    return context

print('Context-expanded KB interpretation function loaded.')

## 6. Shared Criterion Fact Sheet Generation

Generate a single authoritative fact sheet per category that all jurors will use.

In [ ]:
def generate_shared_fact_sheet(category_id: str, chunks: List[Dict[str, Any]], kb: Dict[str, Any], expanded_context: Dict[str, Any]) -> Dict[str, Any]:
    """
    Generate shared criterion fact sheet from chunks and expanded KB context.
    """
    fact_sheet = {
        'category_id': category_id,
        'generated_at': datetime.now().isoformat(),
        'summary': {
            'total_chunks': len(chunks),
            'total_chars': sum(c['char_count'] for c in chunks),
            'unique_documents': len(set(c['filename'] for c in chunks)),
            'document_types': list(set(c['doc_type'] for c in chunks))
        },
        'evidence_inventory': {
            'documents': [],
            'keywords_found': [],
            'mandatory_fields_status': {},
            'dates_extracted': [],
            'signatures_found': [],
            'tables_detected': []
        },
        'criteria_evaluation': [],
        'score_gates_evaluation': {},
        'confidence_indicators': {
            'evidence_coverage': 0.0,
            'keyword_match_rate': 0.0,
            'mandatory_field_completion': 0.0,
            'implementation_proof_strength': 0.0,
            'overall_confidence': 0.0
        }
    }
    
    # Build evidence inventory
    doc_chunks = {}
    for chunk in chunks:
        fname = chunk['filename']
        if fname not in doc_chunks:
            doc_chunks[fname] = []
        doc_chunks[fname].append(chunk)
    
    for fname, fchunks in doc_chunks.items():
        total_chars = sum(c['char_count'] for c in fchunks)
        doc_types = list(set(c['doc_type'] for c in fchunks))
        fact_sheet['evidence_inventory']['documents'].append({
            'filename': fname,
            'chunk_count': len(fchunks),
            'total_chars': total_chars,
            'doc_types': doc_types
        })
    
    # Extract keywords
    all_keywords = expanded_context['explicit_requirements']['keywords']
    all_text = ' '.join(c['text'].lower() for c in chunks)
    for kw in all_keywords:
        if kw.lower() in all_text:
            fact_sheet['evidence_inventory']['keywords_found'].append(kw)
    
    # Evaluate mandatory fields
    mandatory_fields = expanded_context['explicit_requirements']['mandatory_fields']
    for field in mandatory_fields:
        field_lower = field.lower()
        field_parts = field_lower.split()
        matched_chunks = []
        for chunk in chunks:
            chunk_text = chunk['text'].lower()
            if any(part in chunk_text for part in field_parts):
                matched_chunks.append(chunk['chunk_id'])
        
        fact_sheet['evidence_inventory']['mandatory_fields_status'][field] = {
            'status': 'found' if matched_chunks else 'missing',
            'matched_chunks': matched_chunks[:5]
        }
    
    # Extract dates
    date_patterns = [
        r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}',
        r'\d{4}[/-]\d{1,2}[/-]\d{1,2}',
        r'\d{1,2}\s+(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*\s+\d{2,4}'
    ]
    for chunk in chunks[:50]:  # Sample first 50 chunks
        for pattern in date_patterns:
            matches = re.findall(pattern, chunk['text'], re.IGNORECASE)
            for match in matches[:3]:
                fact_sheet['evidence_inventory']['dates_extracted'].append({
                    'date': match if isinstance(match, str) else str(match),
                    'chunk_id': chunk['chunk_id'],
                    'filename': chunk['filename']
                })
    
    # Detect signatures
    sig_keywords = ['signature', 'signed', 'tanda tangan', 'ditandatangani', 'authorized']
    for chunk in chunks[:100]:
        chunk_text_lower = chunk['text'].lower()
        if any(kw in chunk_text_lower for kw in sig_keywords):
            fact_sheet['evidence_inventory']['signatures_found'].append({
                'chunk_id': chunk['chunk_id'],
                'filename': chunk['filename']
            })
    
    # Detect tables
    for chunk in chunks[:100]:
        if '|' in chunk['text'] or 'table' in chunk['text'].lower() or 'tabel' in chunk['text'].lower():
            fact_sheet['evidence_inventory']['tables_detected'].append({
                'chunk_id': chunk['chunk_id'],
                'filename': chunk['filename']
            })
    
    return fact_sheet

print('Shared fact sheet generation function loaded.')

## 7. Criterion Evaluation with Confidence Scoring

In [ ]:
def evaluate_criteria_with_confidence(fact_sheet: Dict[str, Any], expanded_context: Dict[str, Any]) -> Dict[str, Any]:
    """
    Evaluate criteria and calculate confidence scores.
    """
    criteria_results = []
    
    # 1. Document Presence Criterion
    doc_count = len(fact_sheet['evidence_inventory']['documents'])
    if doc_count == 0:
        doc_status = 'fail'
        doc_confidence = 1.0
        doc_score = 0.0
    elif doc_count >= 5:
        doc_status = 'pass'
        doc_confidence = 0.95
        doc_score = 10.0
    elif doc_count >= 3:
        doc_status = 'pass'
        doc_confidence = 0.85
        doc_score = 8.0
    elif doc_count >= 1:
        doc_status = 'partial'
        doc_confidence = 0.70
        doc_score = 5.0
    else:
        doc_status = 'fail'
        doc_confidence = 1.0
        doc_score = 0.0
    
    criteria_results.append({
        'criterion': 'document_presence',
        'status': doc_status,
        'score': doc_score,
        'confidence': doc_confidence,
        'evidence': [d['filename'] for d in fact_sheet['evidence_inventory']['documents'][:5]]
    })
    
    # 2. Keyword Coverage Criterion
    required_keywords = expanded_context['explicit_requirements']['keywords']
    found_keywords = fact_sheet['evidence_inventory']['keywords_found']
    if required_keywords:
        kw_coverage = len(found_keywords) / len(required_keywords)
        if kw_coverage >= 0.8:
            kw_status = 'pass'
            kw_confidence = 0.90
            kw_score = 10.0
        elif kw_coverage >= 0.5:
            kw_status = 'partial'
            kw_confidence = 0.75
            kw_score = 6.0
        elif kw_coverage >= 0.2:
            kw_status = 'partial'
            kw_confidence = 0.60
            kw_score = 4.0
        else:
            kw_status = 'fail'
            kw_confidence = 0.85
            kw_score = 1.0
    else:
        kw_status = 'pass'
        kw_confidence = 0.50
        kw_score = 5.0
    
    criteria_results.append({
        'criterion': 'keyword_coverage',
        'status': kw_status,
        'score': kw_score,
        'confidence': kw_confidence,
        'evidence': found_keywords[:10]
    })
    
    # 3. Mandatory Fields Criterion
    mandatory_fields = expanded_context['explicit_requirements']['mandatory_fields']
    field_statuses = fact_sheet['evidence_inventory']['mandatory_fields_status']
    if mandatory_fields:
        found_count = sum(1 for f in field_statuses.values() if f['status'] == 'found')
        field_coverage = found_count / len(mandatory_fields)
        if field_coverage >= 0.9:
            field_status = 'pass'
            field_confidence = 0.95
            field_score = 10.0
        elif field_coverage >= 0.7:
            field_status = 'pass'
            field_confidence = 0.85
            field_score = 8.0
        elif field_coverage >= 0.5:
            field_status = 'partial'
            field_confidence = 0.70
            field_score = 6.0
        elif field_coverage >= 0.2:
            field_status = 'partial'
            field_confidence = 0.55
            field_score = 3.0
        else:
            field_status = 'fail'
            field_confidence = 0.80
            field_score = 1.0
    else:
        field_status = 'pass'
        field_confidence = 0.50
        field_score = 5.0
    
    criteria_results.append({
        'criterion': 'mandatory_fields',
        'status': field_status,
        'score': field_score,
        'confidence': field_confidence,
        'evidence': [f for f, s in field_statuses.items() if s['status'] == 'found'][:10]
    })
    
    # 4. Implementation Evidence Criterion
    sig_count = len(fact_sheet['evidence_inventory']['signatures_found'])
    date_count = len(fact_sheet['evidence_inventory']['dates_extracted'])
    table_count = len(fact_sheet['evidence_inventory']['tables_detected'])
    
    impl_signals = sig_count + date_count + table_count
    if impl_signals >= 10:
        impl_status = 'pass'
        impl_confidence = 0.90
        impl_score = 10.0
    elif impl_signals >= 5:
        impl_status = 'pass'
        impl_confidence = 0.80
        impl_score = 7.0
    elif impl_signals >= 2:
        impl_status = 'partial'
        impl_confidence = 0.65
        impl_score = 5.0
    else:
        impl_status = 'fail'
        impl_confidence = 0.75
        impl_score = 2.0
    
    criteria_results.append({
        'criterion': 'implementation_evidence',
        'status': impl_status,
        'score': impl_score,
        'confidence': impl_confidence,
        'evidence': [f'signatures: {sig_count}', f'dates: {date_count}', f'tables: {table_count}']
    })
    
    # 5. Temporal Validity Criterion
    if date_count >= 10:
        temp_status = 'pass'
        temp_confidence = 0.85
        temp_score = 10.0
    elif date_count >= 5:
        temp_status = 'pass'
        temp_confidence = 0.75
        temp_score = 7.0
    elif date_count >= 1:
        temp_status = 'partial'
        temp_confidence = 0.60
        temp_score = 5.0
    else:
        temp_status = 'fail'
        temp_confidence = 0.70
        temp_score = 1.0
    
    criteria_results.append({
        'criterion': 'temporal_validity',
        'status': temp_status,
        'score': temp_score,
        'confidence': temp_confidence,
        'evidence': [f'dates_found: {date_count}']
    })
    
    # Calculate overall confidence
    confidence_values = [c['confidence'] for c in criteria_results]
    overall_confidence = sum(confidence_values) / len(confidence_values)
    
    # Calculate weighted score
    weighted_sum = sum(c['score'] * c['confidence'] for c in criteria_results)
    total_weight = sum(c['confidence'] for c in criteria_results)
    raw_score = weighted_sum / total_weight if total_weight > 0 else 0.0
    
    return {
        'criteria': criteria_results,
        'raw_score': raw_score,
        'overall_confidence': overall_confidence,
        'confidence_breakdown': {
            'document_presence': doc_confidence,
            'keyword_coverage': kw_confidence,
            'mandatory_fields': field_confidence,
            'implementation_evidence': impl_confidence,
            'temporal_validity': temp_confidence
        }
    }

print('Criterion evaluation with confidence scoring function loaded.')

## 8. Final Score Calculation with Upward Rounding

In [ ]:
def calculate_final_score(raw_score: float, evidence_present: bool) -> int:
    """
    Apply upward rounding with evidence presence check.
    - 0 = no evidence
    - (0, 3] → 3
    - (3, 6] → 6
    - (6, 10] → 10
    """
    if not evidence_present:
        return 0
    if raw_score <= 0:
        return 0
    elif raw_score <= 3:
        return 3
    elif raw_score <= 6:
        return 6
    else:
        return 10

def check_review_flag(confidence: float, threshold: float = 0.8) -> bool:
    """
    Flag for review if confidence below threshold.
    No automatic downgrade.
    """
    return confidence < threshold

print('Final score calculation functions loaded.')

## 9. Execute Evaluation Across All Categories

In [ ]:
v2_results = []

for cat in tqdm(categories, desc='Processing categories'):
    cat_id = cat['id']
    cat_name = cat['name']
    expected = cat['expected_score']
    
    if VERBOSE:
        print(f'\n{"="*80}')
        print(f'Category {cat_id}: {cat_name}')
        print(f'Expected score: {expected}')
    
    # Load KB and chunks
    kb = load_kb(cat_id)
    chunks = load_category_chunks(cat_id, use_large=USE_LARGE_DB)
    
    if not chunks:
        if VERBOSE:
            print(f'  WARNING: No chunks found in database')
        v2_results.append({
            'category_id': cat_id,
            'category_name': cat_name,
            'expected': expected,
            'chunks': 0,
            'raw_score': 0.0,
            'final_score': 0,
            'confidence': 0.0,
            'requires_review': True,
            'correct': int(expected == 0),
            'criteria': [],
            'fact_sheet': {}
        })
        continue
    
    # Expand KB context
    expanded_context = expand_kb_context(cat_id, kb)
    
    # Generate shared fact sheet
    fact_sheet = generate_shared_fact_sheet(cat_id, chunks, kb, expanded_context)
    
    # Evaluate criteria with confidence
    criteria_eval = evaluate_criteria_with_confidence(fact_sheet, expanded_context)
    
    # Calculate final score
    evidence_present = len(chunks) > 0
    raw_score = criteria_eval['raw_score']
    final_score = calculate_final_score(raw_score, evidence_present)
    
    # Check review flag
    confidence = criteria_eval['overall_confidence']
    requires_review = check_review_flag(confidence, CONFIDENCE_THRESHOLD)
    
    # Check correctness
    correct = int(final_score == expected)
    
    result = {
        'category_id': cat_id,
        'category_name': cat_name,
        'expected': expected,
        'chunks': len(chunks),
        'documents': fact_sheet['summary']['unique_documents'],
        'keywords_found': len(fact_sheet['evidence_inventory']['keywords_found']),
        'mandatory_fields_found': sum(1 for f in fact_sheet['evidence_inventory']['mandatory_fields_status'].values() if f['status'] == 'found'),
        'raw_score': round(raw_score, 2),
        'final_score': final_score,
        'confidence': round(confidence, 3),
        'requires_review': requires_review,
        'correct': correct,
        'criteria': criteria_eval['criteria'],
        'confidence_breakdown': criteria_eval['confidence_breakdown']
    }
    v2_results.append(result)
    
    if VERBOSE:
        print(f'  Chunks: {len(chunks)} | Documents: {fact_sheet["summary"]["unique_documents"]}')
        print(f'  Raw score: {raw_score:.2f} → Final: {final_score}')
        print(f'  Confidence: {confidence:.3f} | Review: {requires_review}')
        print(f'  Expected: {expected} | Predicted: {final_score} | Correct: {correct}')

print(f'\n{"="*80}')
print(f'Processed {len(v2_results)} categories')

In [ ]:
result

## 10. Results Summary

In [ ]:
df_v2 = pd.DataFrame([
    {
        'category_id': r['category_id'],
        'category_name': r['category_name'][:50],
        'expected': r['expected'],
        'chunks': r['chunks'],
        'documents': r.get('documents',0),
        'keywords': r.get('keywords_found', 0),
        'mandatory': r.get('mandatory_fields_found', 0),
        'raw_score': r['raw_score'],
        'final_score': r['final_score'],
        'confidence': r['confidence'],
        'requires_review': r['requires_review'],
        'correct': r['correct']
    }
    for r in v2_results
])

        # 'category_id': cat_id,
        # 'category_name': cat_name,
        # 'expected': expected,
        # 'chunks': len(chunks),
        # 'documents': fact_sheet['summary']['unique_documents'],
        # 'keywords_found': len(fact_sheet['evidence_inventory']['keywords_found']),
        # 'mandatory_fields_found': sum(1 for f in fact_sheet['evidence_inventory']['mandatory_fields_status'].values() if f['status'] == 'found'),
        # 'raw_score': round(raw_score, 2),
        # 'final_score': final_score,
        # 'confidence': round(confidence, 3),
        # 'requires_review': requires_review,
        # 'correct': correct,
        # 'criteria': criteria_eval['criteria'],
        # 'confidence_breakdown': criteria_eval['confidence_breakdown']

display(df_v2)

v2_correct = df_v2['correct'].sum()
v2_accuracy = v2_correct / len(df_v2) * 100
v2_review_count = df_v2['requires_review'].sum()
v2_avg_confidence = df_v2['confidence'].mean()

print(f'\n{"="*80}')
print(f'V2 SCORING ACCURACY: {v2_correct}/{len(df_v2)} ({v2_accuracy:.1f}%)')
print(f'BASELINE ORIGINAL: {baseline["summary"]["original"]["accuracy"]:.1f}%')
if 'enhanced_large' in baseline['summary']:
    print(f'BASELINE ENHANCED: {baseline["summary"]["enhanced_large"]["accuracy"]:.1f}%')
print(f'V1 CALIBRATED: {v1_results["summary"]["calibrated_accuracy"]:.1f}%')
print(f'IMPROVEMENT FROM BASELINE: +{v2_accuracy - baseline["summary"]["original"]["accuracy"]:.1f}%')
print(f'IMPROVEMENT FROM V1: {v2_accuracy - v1_results["summary"]["calibrated_accuracy"]:+.1f}%')
print(f'\nCategories requiring review (confidence < {CONFIDENCE_THRESHOLD}): {v2_review_count}/{len(df_v2)}')
print(f'Average confidence: {v2_avg_confidence:.3f}')
print(f'{"="*80}')

## 11. Comparison: Baseline vs V1 vs V2

In [ ]:
df_comparison = pd.DataFrame([
    {
        'category_id': r['category_id'],
        'expected': r['expected'],
        'baseline_original': baseline_by_id.get(r['category_id'], {}).get('original', {}).get('predicted', 'N/A'),
        'baseline_enhanced': baseline_by_id.get(r['category_id'], {}).get('enhanced_large', {}).get('predicted', 'N/A'),
        'v1_calibrated': v1_by_id.get(r['category_id'], {}).get('calibrated_score', 'N/A'),
        'v2_final': r['final_score'],
        'v2_confidence': r['confidence'],
        'v2_correct': r['correct'],
        'v2_review': r['requires_review']
    }
    for r in v2_results
])

display(df_comparison)

print(f'\nCorrect predictions by method:')
print(f'  Baseline original: {baseline["summary"]["original"]["correct"]}/{baseline["total_categories"]}')
if 'enhanced_large' in baseline['summary']:
    print(f'  Baseline enhanced: {baseline["summary"]["enhanced_large"]["correct"]}/{baseline["total_categories"]}')
print(f'  V1 calibrated:     {v1_results["summary"]["calibrated_correct"]}/{v1_results["total_categories"]}')
print(f'  V2 final:          {v2_correct}/{len(df_v2)}')

## 12. Review Flag Analysis

In [ ]:
df_review = df_v2[df_v2['requires_review'] == True]

print(f'=== CATEGORIES REQUIRING REVIEW (confidence < {CONFIDENCE_THRESHOLD}) ===\n')
if len(df_review) > 0:
    display(df_review[['category_id', 'category_name', 'expected', 'final_score', 'confidence', 'correct']])
else:
    print('No categories require review.')

print(f'\nSummary:')
print(f'  Total flagged: {len(df_review)}/{len(df_v2)}')
print(f'  Flagged but correct: {df_review["correct"].sum()}/{len(df_review)}')
print(f'  Flagged and incorrect: {len(df_review) - df_review["correct"].sum()}/{len(df_review)}')

## 13. Criterion Performance Analysis

In [ ]:
all_criteria = []
for r in v2_results:
    for c in r.get('criteria', []):
        all_criteria.append({
            'category_id': r['category_id'],
            'criterion': c['criterion'],
            'status': c['status'],
            'score': c['score'],
            'confidence': c['confidence']
        })

df_criteria = pd.DataFrame(all_criteria)

print('=== CRITERION PERFORMANCE ACROSS ALL CATEGORIES ===\n')
criterion_stats = df_criteria.groupby('criterion').agg({
    'status': lambda x: (x == 'pass').sum() / len(x) * 100,
    'score': 'mean',
    'confidence': 'mean'
}).round(2)
criterion_stats.columns = ['pass_rate_%', 'avg_score', 'avg_confidence']
display(criterion_stats)

print('\n=== CRITERION STATUS DISTRIBUTION ===\n')
status_counts = df_criteria.groupby('criterion')['status'].value_counts().unstack(fill_value=0)
display(status_counts)

## 14. Save Results

In [ ]:
def default_serialized(obj):
    if isinstance(obj, (np.integer, np.int64)):
        return int(obj)
    if isinstance(obj, (np.floating, np.float64)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return str(obj)

v2_report = {
    'timestamp': datetime.now().isoformat(),
    'total_categories': len(v2_results),
    'configuration': {
        'use_large_db': USE_LARGE_DB,
        'enable_llm_extraction': ENABLE_LLM_EXTRACTION,
        'confidence_threshold': CONFIDENCE_THRESHOLD,
        'scoring_logic': 'upward_rounding'
    },
    'summary': {
        'v2_accuracy': v2_accuracy,
        'v2_correct': v2_correct,
        'baseline_original_accuracy': baseline['summary']['original']['accuracy'],
        'baseline_enhanced_accuracy': baseline['summary'].get('enhanced_large', {}).get('accuracy', 'N/A'),
        'v1_accuracy': v1_results['summary']['calibrated_accuracy'],
        'improvement_from_baseline': v2_accuracy - baseline['summary']['original']['accuracy'],
        'improvement_from_v1': v2_accuracy - v1_results['summary']['calibrated_accuracy'],
        'categories_requiring_review': v2_review_count,
        'average_confidence': v2_avg_confidence
    },
    'results': v2_results
}

report_path = os.path.join(REVIEW_DIR, 'v2_accuracy_improvement_results.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(v2_report, f, indent=2, ensure_ascii=False, default = default_serialized)

csv_path = os.path.join(REVIEW_DIR, 'v2_accuracy_improvement_results.csv')
df_v2.to_csv(csv_path, index=False)

comparison_path = os.path.join(REVIEW_DIR, 'v2_comparison_baseline_v1_v2.csv')
df_comparison.to_csv(comparison_path, index=False)

print(f'Results saved:')
print(f'  JSON: {report_path}')
print(f'  CSV:  {csv_path}')
print(f'  Comparison: {comparison_path}')
print(f'\nFile sizes:')
print(f'  JSON: {os.path.getsize(report_path) / 1024:.1f} KB')
print(f'  CSV:  {os.path.getsize(csv_path) / 1024:.1f} KB')
print(f'  Comparison: {os.path.getsize(comparison_path) / 1024:.1f} KB')

## 15. Key Findings & Next Steps

In [ ]:
print('=== KEY FINDINGS ===\n')
print(f'1. V2 achieved {v2_accuracy:.1f}% accuracy ({v2_correct}/{len(df_v2)} correct)')
print(f'   - Improvement from baseline original: +{v2_accuracy - baseline["summary"]["original"]["accuracy"]:.1f}%')
print(f'   - Improvement from V1: {v2_accuracy - v1_results["summary"]["calibrated_accuracy"]:+.1f}%')
print()
print(f'2. Confidence calibration:')
print(f'   - {v2_review_count}/{len(df_v2)} categories flagged for review (confidence < {CONFIDENCE_THRESHOLD})')
print(f'   - Average confidence: {v2_avg_confidence:.3f}')
print()
print(f'3. Criterion performance:')
for criterion in ['document_presence', 'keyword_coverage', 'mandatory_fields', 'implementation_evidence', 'temporal_validity']:
    crit_data = df_criteria[df_criteria['criterion'] == criterion]
    if len(crit_data) > 0:
        pass_rate = (crit_data['status'] == 'pass').sum() / len(crit_data) * 100
        avg_conf = crit_data['confidence'].mean()
        print(f'   - {criterion}: {pass_rate:.1f}% pass rate, {avg_conf:.3f} avg confidence')
print()
print('=== NEXT STEPS ===\n')
print('1. Review flagged categories (requires_review = True)')
print('2. Expand KB for low-confidence criteria')
print('3. Implement shared fact sheet in production pipeline')
print('4. Integrate confidence-based review flagging')
print('5. Conduct peer-human validation on flagged cases')
print('6. Tune juror prompts to use shared fact sheets')
print('7. Add holdout evaluation before production deployment')